## Imports

In [1]:
# Python standards
import numpy as np
import pandas as pd 
import seaborn as sns
import os
import csv

# Project Specific
import mygene
from Bio import Entrez

## Import Data

In [2]:
# Import data files - be able to differentiate between .csv and txt

# Makes index column (row labels) lowercase and string format
def lowercase_index(table):
    table.index = table.index.map(str)
    table.index = table.index.str.lower()
    return table

# Turn input data into pandas tables
def make_table(filepath):
    filename, extension = os.path.splitext(filepath)

    # Convert .csv file to 
    if extension == ".csv":
        table = pd.read_csv(filepath, header=0, index_col=0) 

    # For non .csv files (.txt, .tsv), use Sniffer to automatically detect delimiter type
    elif extension in [".txt", ".tsv"]:
        with open(filepath, 'r') as f1:
            dialect = csv.Sniffer().sniff(f1.readline())
            delimiter = dialect.delimiter
        table = pd.read_csv(filepath, sep=delimiter, header=0, index_col=0)

    # Raise error for non-supported file types
    else:
        raise TypeError("Unsupported format - file must be .csv, .txt, or .tsv")

    # Convert index (row names) to strings and all lowercase - this makes future processing/matching much easier  
    table = lowercase_index(table)
    table = table.astype(int)
    
    return table

In [3]:
gse167216 = make_table("/home/yaogilbe/CMSE410/cmse410-final-project/GSE167216_Raw_gene_counts_matrix.txt")
gse167216

,I00001,I00002,I00003,I00004,I00005,I00006,I00007,I00008,I00009,I00010,...,I00027,I00028,I00029,I00030,I00031,I00032,I00033,I00034,I00035,I00036
0610007p14rik,858,901,1053,1413,882,704,947,747,1253,928,...,1108,1293,1510,1080,474,1098,857,1187,1572,1111
0610009b22rik,430,515,450,423,395,341,392,449,556,382,...,395,428,515,522,625,487,332,406,440,459
0610009l18rik,14,10,12,9,8,8,12,11,8,5,...,13,7,10,8,10,8,6,7,20,13
0610009o20rik,600,462,559,541,441,268,459,523,594,499,...,475,441,644,479,619,457,437,508,690,378
0610010f05rik,615,439,503,635,448,399,587,592,663,598,...,665,621,733,546,791,572,587,548,701,539
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
mt-nd3,7111,12690,9805,8066,6143,6600,4815,9508,8672,9020,...,6218,9097,6992,5356,10781,8872,8342,8975,6406,3930
mt-nd4,21825,29177,26462,33494,32804,16978,15866,33967,30352,29477,...,33113,29928,24068,16664,35913,29747,25289,41894,30768,16901
mt-nd4l,2540,3058,2926,3801,2663,1873,1980,3266,3416,3179,...,3703,3160,2709,1830,3963,3257,2984,3175,3554,2027
mt-nd5,12329,19694,17297,17376,22612,10367,9914,23960,17992,18814,...,21511,15814,15392,10895,23150,15632,16334,31330,17930,9719


In [4]:
gse130970_raw = make_table("/home/yaogilbe/CMSE410/cmse410-final-project/GSE130970_all_sample_salmon_tximport_counts_entrez_gene_ID.csv")
gse130970_raw

,440349.1.X_1,440350.1.X_1,440351.1.X_4,440352.1.X_4,440353.1.X_4,440354.1.X_4,440355.1.X_4,440357.1.X_5,440375.1.X_8,440376.1.X_1,...,440528.1.X_5,440529.1.X_5,440534.1.X_5,440538.1.X_6,440548.1.X_7,449058.1.X_7,449060.1.X_8,449063.1.X_8,449064.1.X_8,449065.1.X_5
entrez_id,,,,,,,,,,,,,,,,,,,,,
1,15968,15623,12255,13328,6906,10556,9997,10990,14610,8831,...,12238,12415,8589,5733,11558,10913,7998,11437,12920,12134
10,1898,1635,1476,1359,847,2195,1546,2677,1094,1251,...,1058,1173,1389,806,1151,1614,1001,1216,1806,1381
100,100,72,67,70,112,47,76,80,57,37,...,34,49,30,22,41,35,38,59,38,28
1000,2969,2997,2547,2625,3644,2485,1216,2240,2569,2756,...,3035,2525,2624,2017,2851,2290,1854,2906,2652,2712
10000,500,398,526,587,907,327,442,408,500,394,...,521,501,296,326,374,405,439,644,704,436
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9990,747,655,491,579,668,477,212,519,496,663,...,648,601,734,549,691,554,433,637,713,567
9991,2211,1761,2105,1730,3392,1700,1052,1839,1679,1464,...,1203,1795,1140,873,1204,1427,1740,1900,1616,1388
9992,107,10,11,27,5,8,31,11,6,10,...,11,16,15,9,24,82,16,13,19,18


In addition, each dataframe must have an accompanying metadata dataframe, containing information about the `group` (species) and `condition` (disease status) of each sample. Therefore, `.csv` files will be imported that contain this information.

In [5]:
gse130970_md = pd.read_csv("GSE130970_Metadata.csv", header=0, index_col=0)
gse167216_md = pd.read_csv("GSE167216_Metadata.csv", header=0, index_col=0)
gse130970_md

,440349.1.X_1,440350.1.X_1,440351.1.X_4,440352.1.X_4,440353.1.X_4,440354.1.X_4,440355.1.X_4,440357.1.X_5,440375.1.X_8,440376.1.X_1,...,440528.1.X_5,440529.1.X_5,440534.1.X_5,440538.1.X_6,440548.1.X_7,449058.1.X_7,449060.1.X_8,449063.1.X_8,449064.1.X_8,449065.1.X_5
condition,5,4,5,5,6,6,5,6,4,4,...,1,1,0,0,0,6,6,5,3,0
group,human,human,human,human,human,human,human,human,human,human,...,human,human,human,human,human,human,human,human,human,human


## Data formatting

For the dataset GSE130970, the gene IDs are instead recorded as `entrez_id`. For the sake of readability, we want to map these to gene symbols, so we use the `mygene` package to query the Entrez ID system for their matching gene symbols, creating a new pandas dataframe with these symbols.

In [6]:
def convert_entrez(table, genome):
    
    mg = mygene.MyGeneInfo()
    entrez_ID_list = table.index.tolist()
    
    print("Starting MyGene.info query...")
    
    results = mg.querymany(entrez_ID_list,
                           scopes='entrezgene',
                           fields='symbol',
                           species=genome,
                           as_dataframe=True)
    
    # Drop entries without corresponding gene symbols 
    results = results.dropna(subset=['symbol'])
    mapping = results[['symbol']]
    mapping = mapping[~mapping.index.duplicated(keep='first')]
    
    # Find where mapping index intersects with original Entrez ID table and overwrite - replaces labels
    table = table.loc[table.index.intersection(mapping.index)]
    table.index = mapping.loc[table.index, 'symbol']
    
    # Duplicate gene symbols have their expression patterns averaged
    table = table.groupby(table.index).mean()
    
    print(f"Mapped {len(table)} genes to symbols")

    # Convert index (row names) to strings and all lowercase  
    table = lowercase_index(table)
    
    return table

In [7]:
gse130970 = convert_entrez(gse130970_raw, "human")

Starting MyGene.info query...


159 input query terms found no hit:	['100126582', '100127889', '100128374', '100130285', '100132705', '100133144', '100133301', '1001340


Mapped 19426 genes to symbols


In [9]:
# Save the completed tables to .csv files, in case they are needed later

gse130970.to_csv("symbol_datasets/GSE130970_all_sample_counts_gene_symbol.csv")
gse167216.to_csv("symbol_datasets/GSE167216_all_sample_counts_gene_symbol.csv")

We now introduce two more functions, `common_genes` and `subset_by_genes`. `common_genes` finds the intersection of the genes between two given datasets, while `subset_by_genes` produces a dataframe from an existing dataframe, using a list of provided genes as a mask. Combining these two, we can filter existing gene datasets to only contain their intersecting, or shared, genes.  

In [10]:
def common_genes(df1, df2):
    return (df1.index).intersection(df2.index)

def subset_by_genes(df, genes):
    return df.loc[(df.index).intersection(genes)]

In [11]:
commons = common_genes(gse130970, gse167216)
gse130970_commons = subset_by_genes(gse130970, commons)
gse167216_commons = subset_by_genes(gse167216, commons)
gse167216_commons

,I00001,I00002,I00003,I00004,I00005,I00006,I00007,I00008,I00009,I00010,...,I00027,I00028,I00029,I00030,I00031,I00032,I00033,I00034,I00035,I00036
a1bg,166,0,2,5,0,2,57,4,2,1,...,0,1,1,18,2,2,3,2,2,23
a1cf,1298,1713,1498,1891,2637,1337,1396,2199,1940,1896,...,2392,2000,2561,806,1238,2019,1866,3028,2869,901
a2m,7,4,11,6,3,125,2281,7,6,5,...,4,2,10,22576,54,19,8,6,7,2962
a3galt2,0,0,1,0,4,0,4,2,0,0,...,1,0,0,0,7,0,2,0,1,0
a4galt,153,19,22,15,23,8,83,32,20,20,...,28,18,25,19,361,15,25,24,27,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
mt-nd3,7111,12690,9805,8066,6143,6600,4815,9508,8672,9020,...,6218,9097,6992,5356,10781,8872,8342,8975,6406,3930
mt-nd4,21825,29177,26462,33494,32804,16978,15866,33967,30352,29477,...,33113,29928,24068,16664,35913,29747,25289,41894,30768,16901
mt-nd4l,2540,3058,2926,3801,2663,1873,1980,3266,3416,3179,...,3703,3160,2709,1830,3963,3257,2984,3175,3554,2027
mt-nd5,12329,19694,17297,17376,22612,10367,9914,23960,17992,18814,...,21511,15814,15392,10895,23150,15632,16334,31330,17930,9719


In [12]:
gse130970_commons.to_csv("commons_datasets/gse130970_commons.csv")
gse167216_commons.to_csv("commons_datasets/gse167216_commons.csv")

For the next step, differential gene expression analysis, we must break apart our datasets into subsets containing only two groups; a group of interest, and a experimental disease/treatment group. The following function will allow us to make subsets of count and metadata dataframes based on user-specified "conditions" within the metadata.

In [13]:
def subset_by_condition(counts_df, metadata, condition_name, conditions_to_keep):

    # Ensure condition column exists
    if condition_name not in metadata.columns:
        raise ValueError(f"{condition_name} not found in metadata columns")

    # Filter metadata
    subset_metadata = metadata[metadata[condition_name].isin(conditions_to_keep)].copy()
    if subset_metadata.empty:
        raise ValueError("No samples match the specified conditions")

    # Get sample IDs
    sample_ids = subset_metadata.index
    subset_counts = counts_df[sample_ids]
    
    # Align
    subset_metadata = subset_metadata.loc[sample_ids]

    print(f"  Subset complete:")
    print(f"  Samples retained: {len(sample_ids)}")
    print(f"  Conditions: {conditions_to_keep}")
    print(f"  Counts shape: {subset_counts.shape}")

    return subset_counts, subset_metadata

In [14]:
gse130970_commons_sub_06, gse130970_commons_sub_06_md = subset_by_condition(gse130970_commons, gse130970_md.T, "condition", ["0","6"])
gse167216_commons_sub_2, gse167216_commons_sub_2_md = subset_by_condition(gse167216_commons, gse167216_md.T, "condition", ["control_2","ccl4_2"])
gse167216_commons_sub_12, gse167216_commons_sub_12_md = subset_by_condition(gse167216_commons, gse167216_md.T, "condition", ["control_12","ccl4_12"])

  Subset complete:
  Samples retained: 12
  Conditions: ['0', '6']
  Counts shape: (15183, 12)
  Subset complete:
  Samples retained: 12
  Conditions: ['control_2', 'ccl4_2']
  Counts shape: (15183, 12)
  Subset complete:
  Samples retained: 12
  Conditions: ['control_12', 'ccl4_12']
  Counts shape: (15183, 12)


In [15]:
gse130970_commons_sub_06.to_csv("subset_commons_datasets/gse130970_commons_sub_06.csv")
gse130970_commons_sub_06_md.to_csv("subset_commons_datasets/gse130970_commons_sub_06_md.csv")

gse167216_commons_sub_2.to_csv("subset_commons_datasets/gse167216_commons_sub_2.csv")
gse167216_commons_sub_2_md.to_csv("subset_commons_datasets/gse167216_commons_sub_2_md.csv")

gse167216_commons_sub_12.to_csv("subset_commons_datasets/gse167216_commons_sub_12.csv")
gse167216_commons_sub_12_md.to_csv("subset_commons_datasets/gse167216_commons_sub_12_md.csv")

## Differential gene expression analysis

Now, we perform **differential gene expression analysis**. In the original paper, this was done in R using the `limma` package; to perform this in a Python environment, we can instead use the `PyDESeq2` package and pipeline. This starts with importing the `pydeseq2` package and its associated functions.
```bash
pip install pydeseq2
```

In [16]:
# Import pydeseq2 important packages

import pickle as pkl

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
from pydeseq2.utils import load_example_data # not necessary, used for tests

However, the PyDESeq2 package can only perform pairwise comparisons, but there might be multiple different disease or treatment types or states. Therefore, we must write code for Python that will automatically perform pairwise DESeq2 analysis between all conditions against a user-specified control variable.

In [17]:
# Deprecated function for deseq2
def run_deseq2(counts_df, metadata, condition_name, condition_label, control_label):
    """
    Parameters
    ----------
    counts_df : pandas.DataFrame
        Gene expression count matrix formatted as:
            samples × genes
        Values must be raw integer counts (no normalization or log transformation).

    metadata : pandas.DataFrame
        Sample metadata formatted as:
            samples × variables
        Must include a column corresponding to `condition_name` that specifies
        the experimental condition for each sample.

    condition_name : str
        Name of the column in `metadata` that defines experimental groups
        (e.g., "condition").

    condition_label : str
        The condition to compare against the control (e.g., "disease").

    control_label : str
        The reference/control condition (e.g., "control", "healthy").
        Must exist within `metadata[condition_name]`.

    Returns
    -------
    pandas.DataFrame
        A DataFrame containing differential expression results for all genes,
        including:
            - log2FoldChange : log2 fold change (condition vs control)
            - pvalue         : raw p-value
            - padj           : adjusted p-value (FDR)
            - stat           : Wald statistic

    Raises
    ------
    ValueError
        If the specified control label is not found in the metadata.
    """
    
    # SANITY CHECKS
    if counts_df.empty:
        raise ValueError("counts_df is empty")

    if metadata.empty:
        raise ValueError("metadata is empty")

    # Ensure alignment
    if not all(counts_df.index == metadata.index):
        raise ValueError("Sample mismatch: counts_df index != metadata index")

    # Ensure numeric counts
    counts_df = counts_df.apply(pd.to_numeric, errors="coerce")

    if counts_df.isnull().all().all():
        raise ValueError("All count values became NaN after numeric conversion")

    # Fill any remaining NaNs with 0 (safe for counts)
    counts_df = counts_df.fillna(0)

    # Ensure integer counts
    counts_df = counts_df.round().astype(int)

    # Validate control label
    if control_label not in metadata[condition_name].unique():
        raise ValueError("Control label not found in metadata")

    # Default variable for deseq2 process
    inference = DefaultInference(n_cpus=8)

    # Create a dds object for the analysis - code modified from pydeseq2 documentation 
    # https://pydeseq2.readthedocs.io/en/latest/auto_examples/plot_minimal_pydeseq2_pipeline.html#data-loading
    dds = DeseqDataSet(
        counts=counts_df,
        metadata=metadata,
        design="~condition",
        refit_cooks=True,
        inference=inference, # n_cpus=8, # n_cpus can be specified here or in the inference object
    )

    # Run DESeq2 pipeline
    dds.deseq2() # the magic!
    
    stat_res = DeseqStats(dds,contrast=(condition_name, condition_label, control_label))
    stat_res.summary()

    res_df = stat_res.results_df.copy()
    results = res_df # append new creation to results dictionary

    return results

For technical reasons, the differential gene expression algorithm has a built-in memory leak and cannot handle analysis of all 20,000 genes at once. Therefore, we will, for the sake of this project, only perform analysis on the top 100 genes of each dataset. Below is the setup to obtain these datasets.

In [18]:
gse130970_commons_sub_06_t100 = gse130970_commons_sub_06.head(100).copy()
gse167216_commons_sub_2_t100 = gse167216_commons_sub_2.head(100).copy()
gse167216_commons_sub_12_t100 = gse167216_commons_sub_12.head(100).copy()

In [19]:
# DO NOT UNCOMMENT UNLESS YOU WANT TO CRASH THE SERVER

# gse130970_commons_sub_06_DeseqStats_t100 = run_deseq2(gse130970_commons_sub_06_t100.T, gse130970_commons_sub_06_md, condition_name="condition", condition_label="6", control_label="0")
# gse130970_commons_sub_06_DeseqStats_t100.to_csv("t100_DeSeqStats/gse130970_commons_sub_06_DeseqStats_t100.csv")

# gse167216_commons_sub_2_DeseqStats_t100 = run_deseq2(gse167216_commons_sub_2_t100.T, gse167216_commons_sub_2_md, condition_name="condition", condition_label="ccl4_2", control_label="control_2")
# gse167216_commons_sub_2_DeseqStats_t100.to_csv("t100_DeSeqStats/gse167216_commons_sub_2_DeseqStats_t100.csv")

# gse167216_commons_sub_12_DeseqStats_t100 = run_deseq2(gse167216_commons_sub_12_t100.T, gse167216_commons_sub_12_md, condition_name="condition", condition_label="ccl4_12", control_label="control_12")
# gse167216_commons_sub_12_DeseqStats_t100.to_csv("t100_DeSeqStats/gse167216_commons_sub_12_DeseqStats_t100.csv")